# Tile QC — EMIT Cloud Mask

Runs after `Color_Matching.ipynb`. Downloads EMIT L2A mask NCs, reprojects
cloud/cirrus flags to the same UTM grid, flags tiles by cloud fraction + S2 brightness.
All thresholds come from `pipeline_config.yaml`.

In [ ]:
import os, textwrap
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git
%cd /content/HyperSpectralSuperResolution/

In [ ]:
!pip install -q numpy scipy rasterio matplotlib tqdm earthaccess netCDF4 pyproj shapely xarray python-dateutil pyyaml

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import earthaccess as ea
ea.login(strategy="environment")

In [ ]:
DRIVE_BASE = "/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22"

!python data/tile_qc.py --drive-base "$DRIVE_BASE"

## Diagnostics

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

drive_base = Path(DRIVE_BASE)
with open(drive_base / "pipeline_config.yaml") as f:
    cfg = yaml.safe_load(f)

max_cloud = cfg["qc_max_emit_cloud_frac"]
max_bright = cfg["qc_max_s2_bright_frac"]

qc_df = pd.read_csv(drive_base / "r2_tiles_qc.csv")
n_pass = qc_df["pass_qc"].sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].hist(qc_df["combined_frac"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[0].axvline(max_cloud, color="red", ls="--", label=f"threshold={max_cloud}")
axes[0].set_xlabel("EMIT cloud+cirrus fraction")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(qc_df["s2_bright_frac"].dropna(), bins=50, edgecolor="black", alpha=0.7)
axes[1].axvline(max_bright, color="red", ls="--", label=f"threshold={max_bright}")
axes[1].set_xlabel("S2 bright pixel fraction")
axes[1].set_ylabel("Count")
axes[1].legend()

c = qc_df["pass_qc"].map({True: "green", False: "red"})
axes[2].scatter(qc_df["r2_mean"], qc_df["combined_frac"], alpha=0.3, s=5, c=c)
axes[2].set_xlabel("Mean R²")
axes[2].set_ylabel("Cloud+cirrus fraction")

plt.tight_layout()
fig.savefig(str(drive_base / "qc_diagnostics.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"Final: {n_pass}/{len(qc_df)} tiles pass QC")